# Compare embedding distribution
This notebook computes the model embeddings for two data sets and visualises their distributions with t-SNE.

In [ ]:
import pandas as pd
from gnn import create_tf_dataset, CustomPreprocessor, atom_features, bond_features, global_features
import tensorflow as tf
import nfp
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the data
df_aug = pd.read_csv('data/Aug-DBs/Aug-DB1.csv')
# dff2 path comes from notebooks/data.py.ipynb
dff2 = pd.read_csv('/home/nanta/Redox_mediator_screening/data/Filtered_data/filter2.csv.gz', compression='gzip')
df_aug = df_aug[['can_smiles_solute','can_smiles_solvent','DGsolv']]
dff2 = dff2[['can_smiles_solute','can_smiles_solvent','DGsolv']]

In [ ]:
# Load model and preprocessor
model = tf.keras.models.load_model('model_files/SSD_models/student35/best_model.h5', custom_objects=nfp.custom_objects)
preprocessor = CustomPreprocessor(explicit_hs=False, atom_features=atom_features, bond_features=bond_features)
preprocessor.from_json('model_files/SSD_models/student35/preprocessor.json')

extractor = tf.keras.Model(model.inputs, [model.layers[-1].input])
output_signature = (preprocessor.output_signature, tf.TensorSpec(shape=(), dtype=tf.float32), tf.TensorSpec(shape=(), dtype=tf.float32))

def embed_dataframe(df):
    ds = tf.data.Dataset.from_generator(
        lambda: create_tf_dataset(df, preprocessor, 1.0, False),
        output_signature=output_signature
    ).padded_batch(batch_size=len(df))
    return extractor.predict(ds).squeeze()

emb_aug = embed_dataframe(df_aug)
emb_dff2 = embed_dataframe(dff2)

In [ ]:
# t-SNE comparison
all_emb = np.concatenate([emb_aug, emb_dff2])
labels = np.array([0]*len(emb_aug) + [1]*len(emb_dff2))
pipe = Pipeline(steps=[('PCA', PCA(n_components=10)), ('TSNE', TSNE(n_components=2, random_state=0))])
tsne_coords = pipe.fit_transform(all_emb)
plt.figure(figsize=(6,5))
plt.scatter(tsne_coords[labels==0,0], tsne_coords[labels==0,1], s=10, label='Aug-DB1')
plt.scatter(tsne_coords[labels==1,0], tsne_coords[labels==1,1], s=10, label='dff2')
plt.legend()
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('Embedding distribution comparison')
plt.show()